# 集合通信原语与 partition notation

> 上一节看了 DDP、ZeRO、FSDP 这些并行方案，它们都在反复调用同一组底层通信算子：all-reduce 把梯度平均、all-gather 把分片参数拼起来、reduce-scatter 把梯度切片后聚合。框架把这些细节藏起来，但读懂任何一篇并行论文——Megatron、DeepSpeed、ZeRO、SP/TP/EP——的前提是看懂这组算子在做什么。
>
> 这一节从最基础的点对点 send/recv 出发，逐个推导七个 collective operations 的语义和通信量，引入 partition notation 这套符号系统描述「张量沿哪个维度切」，最后落到 torch.distributed 的真实 API 上，用 gloo backend 在两个进程上跑一遍。


集合通信（collective communication）指的是一组进程共同参与的通信操作，区别于两个进程之间的点对点通信。在分布式训练里，每个 GPU 对应一个进程，它们通过 NCCL 或 gloo 这类底层库交换数据。理解集合通信的关键不是记住 API 签名，而是搞清楚每个操作前后张量在哪些设备上、形状如何变化、通信量是多少 bytes。

描述「张量在多卡上怎么分布」需要一套符号。这一节采用 partition notation：把设备网格（device mesh）的轴命名为 $X$、$Y$，把张量的轴命名为 $I$、$J$、$K$，用下标 $I_X$ 表示「张量的 $I$ 轴沿设备网格的 $X$ 轴切分」。这套符号出现在 Megatron-LM、Alpa、MaxText、DeepSpeed 几乎所有并行论文里，是讨论分布式计算的标准语言。

本节的代码用 numpy 在单进程里模拟多卡——每个 numpy 数组代表一张卡上的数据，通过显式地拼接和求和演示 collective 的语义。等语义清楚后，最后一节切到 torch.distributed，在真实的两个进程上跑 all-reduce。


## 1. 点对点通信：send 和 recv

集合通信的地基是两个进程之间的点对点通信。PyTorch 提供两个最底层的 API：`send` 把一个 tensor 发给目标进程，`recv` 从源进程接收一个 tensor。它们的语义是阻塞的——`send` 在对方收到之前不返回，`recv` 在数据到达之前不返回。

阻塞通信的问题是容易死锁。进程 A 先 `send` 给 B 再 `recv` 从 B，进程 B 也先 `send` 给 A 再 `recv` 从 A，两边都在等对方接收，谁也走不下去。解决方案是非阻塞版本 `isend` / `irecv`，它们立即返回一个 work handle，后续调用 `work.wait()` 时才真正等待，期间可以干别的活。

实际训练脚本里几乎不会直接用 send / recv。它们出现在流水并行（Pipeline Parallelism）里：每个 stage 把 hidden state 发给下一个 stage。集合通信——all-reduce、all-gather 这些——才是数据并行和模型并行的主力，下面用 numpy 把它们一个一个拆开看。


在调用任何 collective 之前，进程必须先加入一个 process group。process group 是一组互相能通信的进程的集合，每个进程在组内有一个 rank（编号从 0 开始）和一个 world_size（组的总进程数）。

为什么需要这个概念？因为大模型训练常常混用多种并行：64 张卡可能分成 8 个数据并行组、每组 8 张卡做 ZeRO，组内的 all-reduce 不应该让别的组的卡参与。process group 让我们指定「这次通信在哪个子集内发生」。`dist.new_group(ranks=[0,1,2,3])` 就能创建一个子组，传给 collective 的 `group` 参数即可限定范围。

下面开始具体算子。每个算子都用一个 4 卡的 numpy mock 演示，每个 numpy 数组代表一张卡持有的数据。


## 2. broadcast：1 → all

broadcast 是最简单的 collective。rank 0 持有一份数据，把它复制到所有其他卡上。语义是「一份输入，N 份相同输出」。

典型场景：训练开始前把模型参数从 rank 0 广播到所有卡，让每张卡的初始权重一致。DDP 的 `model = DDP(model)` 内部第一步就是 broadcast parameters。


In [ ]:
# === broadcast 的 numpy mock ===
import numpy as np

# 假设有 4 张卡，rank 0 持有原始数据
np.random.seed(0)
data_on_rank0 = np.array([10, 20, 30, 40])

# broadcast 前：只有 rank 0 有数据，其他卡是空的
cards_before = [None, None, None, None]
cards_before[0] = data_on_rank0.copy()
print("broadcast 前：")
for rank, data in enumerate(cards_before):
    print(f"  rank {rank}: {data}")

# broadcast 后：每张卡都有 rank 0 的副本
cards_after = [data_on_rank0.copy() for _ in range(4)]
print("\nbroadcast 后：")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\n关键观察：1 份输入 → 4 份相同输出，通信量 = 数据大小 × (N-1)。")


## 3. scatter：1 → all（数据切片）

scatter 和 broadcast 的区别：rank 0 持有 N 份数据，把第 i 份发给第 i 张卡。语义是「一份大数组，切成 N 片分发」。

典型场景：rank 0 读取了一个 batch 的数据，切成 N 份分给 N 张卡做数据并行。每张卡拿到的是 batch 的不同子集。


In [ ]:
# === scatter 的 numpy mock ===
import numpy as np

np.random.seed(0)
# rank 0 持有一份数据，准备切成 4 份
big_array = np.array([[1, 2],
                      [3, 4],
                      [5, 6],
                      [7, 8]])

# scatter 前：rank 0 持有完整数据
print("scatter 前：")
print(f"  rank 0 (完整):\n{big_array}")
for rank in range(1, 4):
    print(f"  rank {rank}: 空")

# scatter：rank 0 沿 axis=0 切成 4 份，第 i 份发给第 i 张卡
shards = np.split(big_array, 4, axis=0)
print("\nscatter 后：")
for rank, shard in enumerate(shards):
    print(f"  rank {rank}: {shard.ravel()}")

print("\n关键观察：1 份大数组 → N 份不同切片，每卡拿到 1/N 的数据。")
print("和 broadcast 的区别：broadcast 每卡拿到相同副本，scatter 每卡拿到不同切片。")


## 4. gather：all → 1

gather 是 scatter 的反操作。每张卡持有一片数据，rank 0 把所有片按顺序拼成一份完整数组。语义是「N 份输入，1 份拼接输出」。

典型场景：分布式推理时把每张卡生成的 token 序列收到 rank 0 做后处理。注意 gather 只在 rank 0 上有完整结果，其他卡的缓冲区不变。


In [ ]:
# === gather 的 numpy mock ===
import numpy as np

# 每张卡持有一片数据
shards = [np.array([10, 20]),
          np.array([30, 40]),
          np.array([50, 60]),
          np.array([70, 80])]

print("gather 前：")
for rank, s in enumerate(shards):
    print(f"  rank {rank}: {s}")

# gather：rank 0 收集所有片，沿 axis=0 拼接
gathered_on_rank0 = np.concatenate(shards, axis=0)
print("\ngather 后：")
print(f"  rank 0 (拼接): {gathered_on_rank0}")
for rank in range(1, 4):
    print(f"  rank {rank}: 仍持有原数据 {shards[rank]}（未拼接）")

print("\n关键观察：N 份输入 → 1 份拼接输出，只在 rank 0 上。")


## 5. reduce：all → 1（聚合）

reduce 在 gather 的基础上多做一步：把收集到的数据按某种算子（通常是 sum）聚合。rank 0 持有最终的聚合结果，其他卡没有。

典型场景：把每张卡算出的梯度加起来得到平均梯度——但这个场景下每张卡都需要结果，所以用的是下一节的 all-reduce 而不是 reduce。


In [ ]:
# === reduce 的 numpy mock ===
import numpy as np

# 每张卡持有一个 tensor，准备 sum 起来
cards = [np.array([1.0, 2.0]),
         np.array([3.0, 4.0]),
         np.array([5.0, 6.0]),
         np.array([7.0, 8.0])]

print("reduce 前：")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# reduce sum：rank 0 上得到所有卡对应位置的求和
result_on_rank0 = np.sum(np.stack(cards), axis=0)
print("\nreduce sum 后：")
print(f"  rank 0: {result_on_rank0}")
for rank in range(1, 4):
    print(f"  rank {rank}: 未保留结果")

print("\n关键观察：N 份输入 → 1 份聚合输出，只在 rank 0 上。")
print("如果改用 max 算子，结果就是逐位置取最大值。")


## 6. all-reduce：all → all（聚合后广播）

all-reduce 是分布式训练里出现频率最高的 collective。它在 reduce 的基础上把结果广播到所有卡——每张卡都有相同的聚合结果。语义上等价于「先 reduce 到 rank 0，再 broadcast 到所有卡」。

DDP 在反向传播结束时调用一次 all-reduce，让每张卡拿到相同的平均梯度，从而每张卡的优化器 step 后权重仍然一致。

all-reduce 的核心价值在于它的实现算法——ring all-reduce——能让通信量与设备数 N 无关，而不是朴素实现的 $O(N)$ 倍。


In [ ]:
# === all-reduce 的 numpy mock ===
import numpy as np

cards = [np.array([1.0, 2.0]),
         np.array([3.0, 4.0]),
         np.array([5.0, 6.0]),
         np.array([7.0, 8.0])]

print("all-reduce 前：")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# all-reduce sum：每张卡都得到所有卡对应位置的求和
total = np.sum(np.stack(cards), axis=0)
cards_after = [total.copy() for _ in range(4)]

print("\nall-reduce sum 后：")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\n关键观察：N 份不同输入 → N 份相同聚合输出，每张卡都有完整结果。")
print("朴素实现通信量 = N × 数据大小（rank 0 收集再广播）。")
print("ring 算法把这个数字降到 2 × (N-1)/N × 数据大小，与 N 几乎无关。")


### 6.1 ring all-reduce：为什么通信量与设备数无关

朴素 all-reduce 的实现是「rank 0 收集所有数据，求和后广播」。问题在于 rank 0 成了瓶颈：N 张卡都给它发数据，它的入口带宽被打满，通信时间和 N 成正比。

ring all-reduce 把 N 张卡排成一个环。每张卡只和左右邻居通信，分两个阶段：

**阶段一 reduce-scatter**：每张卡把自己的数据切成 N 片。第 $i$ 轮，卡 $r$ 把自己当前的第 $(r - i) \bmod N$ 片发给右邻居，同时从左邻居收到对应片累加到自己。N-1 轮后，每张卡手上有一片是「所有卡那一片的和」——这一片就是最终 reduce 结果的一个切片。

**阶段二 all-gather**：每张卡把自己手上那片最终结果沿环传一圈，N-1 轮后所有卡都有完整的 N 片。

每个阶段每张卡发送 $(N-1)$ 次，每次发 $\text{数据大小}/N$。总通信量 = $2 \times (N-1)/N \times \text{数据大小}$，当 N 较大时约等于 $2 \times \text{数据大小}$，与 N 无关。

下面用 N=4 手算一次完整的 ring all-reduce。


In [ ]:
# === ring all-reduce 演示（N=4）===
# 把环算法拆成两个阶段，每阶段 N-1 轮，每轮每卡只发 1 个元素。
import numpy as np

N = 4
np.random.seed(42)
cards = [list(np.random.randint(1, 10, size=N)) for _ in range(N)]
print("初始：每张卡持有长度 4 的 vector（切成 4 片，每片 1 个元素）")
for r in range(N):
    print(f"  rank {r}: {cards[r]}")

expected = [sum(cards[r][k] for r in range(N)) for k in range(N)]
print(f"\n期望最终结果（每卡都拿到）：{expected}")

# === 阶段一 reduce-scatter ===
# 约定：rank r 在第 step 轮（step=0,...,N-2）：
#   发送片 (r - step) % N 给右邻居 (r+1) % N
#   从左邻居 (r-1) % N 接收片 (r - step - 1) % N，累加到自己的对应位置
# N-1 轮后：rank r 持有片 (r+1) % N 的全局和。
buffer = [row[:] for row in cards]

print("\n--- 阶段一 reduce-scatter ---")
for step in range(N - 1):
    # 同步通信：先记录所有发送值，再统一累加
    outgoing = {}
    for r in range(N):
        send_idx = (r - step) % N
        outgoing[r] = (send_idx, buffer[r][send_idx])
    for r in range(N):
        send_idx, value = outgoing[r]
        right = (r + 1) % N
        buffer[right][send_idx] += value
    print(f"  轮 {step+1} 后：")
    for r in range(N):
        print(f"    rank {r}: {buffer[r]}")

# 验证阶段一
print("\n阶段一验证：rank r 持有片 (r+1)%N 的全局和")
for r in range(N):
    master_idx = (r + 1) % N
    got = buffer[r][master_idx]
    want = expected[master_idx]
    assert got == want, f"rank {r} 片 {master_idx} = {got}，期望 {want}"
    print(f"  rank {r} 片 {master_idx} = {got} OK")

# === 阶段二 all-gather ===
# 每张卡把自己持有的「全局和片」沿环传一圈。
# 每张卡维护一个「当前持有的全局和片索引」，每轮把它发给右邻居，右邻居覆盖对应位置。
# rank r 初始持有的全局和片索引 = (r+1) % N。
# step=0：rank r 发片 (r+1)%N 给 (r+1)%N；右邻居把这个值放到自己的片 (r+1)%N 位置。
#         然后右邻居「当前持有的全局和片」就变成了刚刚收到的那一片（索引仍是 (r+1)%N，但来源是左邻居）。
# step=1：rank r 把「上一轮收到的片」发给右邻居。
# 用一个简单方式：每轮 rank r 从左邻居收到一个 (片索引, 值)，写到自己的对应位置，下一轮再发出去。
print("\n--- 阶段二 all-gather ---")
# 每张卡当前「要发出去的」片（初始是自己的全局和片）
out_chunk_idx = [(r + 1) % N for r in range(N)]
out_value = [buffer[r][(r + 1) % N] for r in range(N)]

for step in range(N - 1):
    # 同步：每张卡把自己的 out 发给右邻居
    snapshot_idx = out_chunk_idx[:]
    snapshot_val = out_value[:]
    for r in range(N):
        right = (r + 1) % N
        # 右邻居收到 (片索引, 值)
        received_idx = snapshot_idx[r]
        received_val = snapshot_val[r]
        # 写入右邻卡的对应位置
        buffer[right][received_idx] = received_val
        # 右邻居下一轮要把这个收到的片继续传出去
        out_chunk_idx[right] = received_idx
        out_value[right] = received_val
    print(f"  轮 {step+1} 后：")
    for r in range(N):
        print(f"    rank {r}: {buffer[r]}")

print("\n最终验证：每张卡是否都拿到了完整的全局和？")
all_ok = True
for r in range(N):
    ok = buffer[r] == expected
    all_ok = all_ok and ok
    print(f"  rank {r}: {buffer[r]}  [{'OK' if ok else 'FAIL'}]")

assert all_ok
print(f"\n所有 rank 都拿到了 {expected}")
print("\n关键观察：")
print(f"  阶段一 {N-1} 轮，每轮每卡发 1 个元素（= D/N）。")
print(f"  阶段二 {N-1} 轮，每轮每卡发 1 个元素（= D/N）。")
print(f"  每卡总通信量 = 2 × (N-1) × (D/N) = 2 × {N-1} × 1 = {2*(N-1)} 个元素。")
print(f"  朴素实现 rank 0 要收 {N} 个、发 {N} 个 = {2*N} 个元素，是瓶颈。")
print(f"  ring 把负载分散到每张卡，与 N 几乎无关。")


## 7. all-gather：all → all（拼接）

all-gather 是 gather 的全连接版本。每张卡持有一片数据，操作后每张卡都拿到所有片的完整拼接。语义是「N 份输入 → N 份相同拼接输出」。

all-gather 的通信量分析同样适用 ring 算法：$(N-1)/N \times \text{数据大小}$，与 N 几乎无关。

典型场景：FSDP 在前向传播前用 all-gather 把分片的参数拼成完整 layer，反向传播用完后丢弃。ZeRO Stage 3 的核心通信就是 all-gather。


In [ ]:
# === all-gather 的 numpy mock ===
import numpy as np

# 每张卡持有一片
shards = [np.array([10, 20]),
          np.array([30, 40]),
          np.array([50, 60]),
          np.array([70, 80])]

print("all-gather 前：")
for rank, s in enumerate(shards):
    print(f"  rank {rank}: {s}")

# all-gather：每张卡都得到所有片的拼接
full = np.concatenate(shards, axis=0)
cards_after = [full.copy() for _ in range(4)]

print("\nall-gather 后：")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\n关键观察：和 gather 的区别——每张卡都有完整结果，不只 rank 0。")
print("ring all-gather 通信量 = (N-1)/N × 数据大小。")


## 8. reduce-scatter：all → all（聚合后切片）

reduce-scatter 是 reduce 的全连接版本，也是 all-reduce 的「前半段」。每张卡持有一份完整数据，操作后每张卡拿到聚合结果的一个切片。语义是「N 份完整输入 → N 份聚合切片输出」。

reduce-scatter 和 all-gather 互为对偶：reduce-scatter 加上分片，all-gather 去掉分片。两者组合起来就是一次 all-reduce。

典型场景：ZeRO Stage 2 在反向传播时用 reduce-scatter，每张卡只保留自己负责的那一片参数对应的梯度，其余梯度立即释放。


In [ ]:
# === reduce-scatter 的 numpy mock ===
import numpy as np

# 每张卡持有一份完整 vector（长度 = N）
N = 4
cards = [np.array([1.0, 2.0, 3.0, 4.0]),
         np.array([10., 20., 30., 40.]),
         np.array([100., 200., 300., 400.]),
         np.array([0.1, 0.2, 0.3, 0.4])]

print("reduce-scatter 前：")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# 先 sum 所有卡，再切成 N 片分给各卡
stacked = np.stack(cards)   # shape (N, N)
summed = stacked.sum(axis=0)  # shape (N,) 每个位置都是所有卡求和
shards = np.split(summed, N)   # 切成 N 份

print("\n所有卡求和后：", summed)
print("\nreduce-scatter 后：")
for rank, shard in enumerate(shards):
    print(f"  rank {rank}: {shard}")

print("\n关键观察：N 份完整输入 → N 份聚合切片输出，每卡只有 1/N 的聚合结果。")
print("ring reduce-scatter 通信量 = (N-1)/N × 数据大小。")


## 9. all-to-all：all ↔ all（数据重排）

all-to-all 是最复杂的 collective。每张卡持有 N 份不同目标的数据——第 i 份是要发给第 i 张卡的。操作后，每张卡收到的数据来自所有其他卡的第 r 份（r 是自己的 rank）。语义是「数据沿 rank 维度做转置」。

all-to-all 的特点是通信模式高度非对称：每张卡发给每张卡的数据都不同。这让它无法套用 ring 算法的高效实现，通信量随 N 平方级增长。

典型场景：MoE（Mixture of Experts）的 Expert Parallelism——每张卡持有几个 expert，token 要根据路由分发到对应 expert 所在的卡上。下一节会专门讲。


In [ ]:
# === all-to-all 的 numpy mock ===
import numpy as np

N = 4
# 每张卡持有 N 份 buffer，第 i 份要发给 rank i
# 用 (sender, receiver) 二维表看清楚
send_buffers = [
    # rank 0 持有的 4 份，第 0 份给自己，第 i 份给 rank i
    np.array([[0, 0], [0, 1], [0, 2], [0, 3]]),  # 发给 0,1,2,3
    np.array([[1, 0], [1, 1], [1, 2], [1, 3]]),
    np.array([[2, 0], [2, 1], [2, 2], [2, 3]]),
    np.array([[3, 0], [3, 1], [3, 2], [3, 3]]),
]

print("all-to-all 前（每卡的发送 buffer，第 i 行发给 rank i）：")
for rank, buf in enumerate(send_buffers):
    print(f"  rank {rank} send_buf:\n{buf}")

# all-to-all 等价于把整个二维表做转置
# rank r 收到的第 j 行 = 来自 rank j 的第 r 行
all_data = np.stack(send_buffers)   # shape (N, N, 2)
transposed = np.transpose(all_data, (1, 0, 2))   # (receiver, sender, 2)

print("\nall-to-all 后（每卡的接收 buffer，第 j 行来自 rank j）：")
for rank in range(N):
    print(f"  rank {rank} recv_buf:\n{transposed[rank]}")

print("\n关键观察：数据在 sender × receiver 二维网格上做转置。")
print("每张卡发给每张卡的数据都不同，不能用 ring 算法高效实现。")
print("通信量 = N × 数据大小（每张卡发出 N 份不同数据）。")


### 七个 collective 总览

把七个算子的输入输出关系和典型场景放在一起：


In [ ]:
# === 七个 collective 对比表 ===
rows = [
    ("算子", "方向", "通信量", "典型场景"),
    ("broadcast",   "1 → all (相同)",    "(N-1) × D",        "初始化时广播参数"),
    ("scatter",     "1 → all (切片)",    "(N-1) × D/N",      "rank 0 分发 batch"),
    ("gather",      "all → 1",           "(N-1) × D/N",      "rank 0 收集结果"),
    ("reduce",      "all → 1 (聚合)",    "(N-1) × D/N",      "rank 0 聚合 loss"),
    ("all-reduce",  "all → all (聚合)",  "2(N-1)/N × D",     "DDP 反向同步梯度"),
    ("all-gather",  "all → all (拼接)",  "(N-1)/N × D",      "FSDP 拼参数、ZeRO-3"),
    ("reduce-scatter", "all → all (聚合切片)", "(N-1)/N × D", "ZeRO-2 切梯度"),
    ("all-to-all",  "all ↔ all (重排)",  "N × D",            "MoE EP 路由 token"),
]
widths = [18, 24, 22, 30]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "算子":
        print("-" * len(line))

print("\nD = 单卡数据大小，N = 设备数。")
print("ring 算法让前 6 个的通信量与 N 几乎无关；all-to-all 无法用 ring 优化。")


## 10. partition notation：描述「张量怎么切」

前面几节用自然语言描述「每张卡持有完整数据」「每张卡持有 1/N 切片」足够清楚，但读到 Megatron、Alpa、MaxText 的论文时，会看到一种紧凑的符号系统——partition notation。它的核心是给设备网格和张量分别命名轴，然后用下标表示「张量的哪个轴沿设备的哪个轴切分」。

**设备网格（device mesh）**：把 N 张卡摆成一个二维网格，命名两个轴 $X$ 和 $Y$。比如 8 张卡可以摆成 $X=2, Y=4$ 的网格，每张卡有一个 $(x, y)$ 坐标。

**张量轴**：矩阵 $A$ 的轴命名为 $I$（行）和 $J$（列）。

**下标规则**：$A[I_X, J]$ 表示矩阵 $A$ 的 $I$ 轴沿设备网格的 $X$ 轴切分——也就是说，$I$ 轴被 $X$ 个设备平分，每个设备持有 $I/X$ 行。$J$ 没有下标，表示 $J$ 轴不切分，每张卡持有完整的 $J$ 列。

常见的几种写法：

- $A[I_X, J]$：行沿 $X$ 切，列复制。$Y$ 不出现表示沿 $Y$ 复制。
- $A[I, J_Y]$：列沿 $Y$ 切，行复制。
- $A[I_{XY}, J]$：行沿「摊平的 XY 网格」切，相当于把 $X \times Y$ 个设备当成一维。
- $A[I, J]$：不切分，每张卡都有完整副本（即 replicated）。


In [ ]:
# === partition notation 的具体张量例子 ===
import numpy as np

# 假设有一个 4×4 的矩阵 A，要放到 2×2 = 4 张卡的网格上
# 设备网格：X 轴 2 个，Y 轴 2 个
A = np.arange(16).reshape(4, 4)
print("完整矩阵 A (shape = (4, 4)):")
print(A)
print()

# 情况 1: A[I_X, J] —— 行沿 X 切，列复制
# X 轴有 2 个值（0 和 1），所以行切成 2 份，每份 2 行
# 沿 Y 的每个设备都拿到相同的行子集
print("=== A[I_X, J]: 行沿 X 切，列沿 Y 复制 ===")
row_shards = np.split(A, 2, axis=0)
for x in range(2):
    for y in range(2):
        print(f"  device (x={x}, y={y}):\n{row_shards[x]}")

print("\n=== A[I, J_Y]: 列沿 Y 切，行沿 X 复制 ===")
col_shards = np.split(A, 2, axis=1)
for x in range(2):
    for y in range(2):
        print(f"  device (x={x}, y={y}):\n{col_shards[y]}")

print("\n关键观察：下标出现的轴 = 切分；不出现的轴 = 复制。")
print("同一份数据，切法不同，每张卡上看到的子矩阵完全不同。")


### 10.1 用 partition notation 描述 collective

partition notation 让 collective 的语义可以用一行公式写清楚。下面是三个核心算子的 notation 形式：

- **all-gather（去 Y 切分）**：$\text{AllGather}_Y: A[I, J_Y] \to A[I, J]$。输入是列沿 Y 切的，输出是完整的——all-gather 把 $J_Y$ 这个切分去掉了。
- **reduce-scatter（加 Y 切分）**：$\text{ReduceScatter}_{Y, J}: A[I, J]\{U_Y\} \to A[I, J_Y]$。输入是「沿 Y 未 reduce」的（标记 $\{U_Y\}$），输出是 reduce 后再沿 Y 切。
- **all-reduce**：$\text{AllReduce}_Y: A[I, J]\{U_Y\} \to A[I, J]$。等价于先 reduce-scatter 再 all-gather。

记号 $\{U_Y\}$ 表示「沿 Y 轴未 reduce」——每张卡都有一个局部值，需要跨 Y 求和才得到真实结果。这个记号在描述 tensor parallelism 的 partial sum 时特别有用。


## 11. 分块 matmul 的四种通信 case

partition notation 真正的价值在于描述 matmul 在并行下的通信需求。考虑矩阵乘法 $C = A \cdot B$，其中 $A$ 是 $I \times J$，$B$ 是 $J \times K$。根据 $A$ 和 $B$ 沿 contracting 维度 $J$ 是否切分，能分成四种 case：

**Case 1：contracting 维度都不切分**

$$A[I_X, J] \cdot B[J, K_Y] \to C[I_X, K_Y]$$

$A$ 的 $J$ 不切，$B$ 的 $J$ 也不切，每个设备直接做局部 matmul，输出天然就是想要的切分方式。**零通信**。这是 tensor parallelism 里 column parallel 输入 + row parallel 输出的理想 case。

**Case 2：一方 contracting 维度切分**

$$A[I, J_X] \cdot B[J, K] \to C[I, K]$$

$A$ 的 $J$ 沿 X 切了，但 $B$ 的 $J$ 没切，无法直接相乘。需要先 all-gather 把 $A$ 的 $J$ 切分去掉，再和 $B$ 做局部 matmul：

$$\text{AllGather}_X: A[I, J_X] \to A[I, J], \quad A[I, J] \cdot B[J, K] \to C[I, K]$$

**通信：一次 all-gather，通信量 = $A$ 的总大小**。这是 FSDP 在前向时 gather 参数的 case。

**Case 3：双方 contracting 维度都切分**

$$A[I, J_X] \cdot B[J_X, K] \to C[I, K]\{U_X\}$$

可以局部相乘，但每个设备得到的是 partial sum——只是 X 个分片里自己那部分。需要 all-reduce 把所有设备的 partial sum 加起来：

$$\text{AllReduce}_X: C[I, K]\{U_X\} \to C[I, K]$$

**通信：一次 all-reduce，通信量 = $C$ 的总大小 × 2(N-1)/N**。这是 tensor parallelism 里 row parallel 的 case。

**Case 4：双方 non-contracting 维度沿同轴切分**

$A$ 的 $I$ 沿 X 切，$B$ 的 $K$ 也沿 X 切——这种情况下 matmul 在每个设备上独立进行，但语义上其实想要的是不同设备的输出，通信需求取决于后续用途。多数情况下等价于 Case 1 的扩展形式，可以做到零通信。


In [ ]:
# === 四种 case 的通信量对比 ===
rows = [
    ("Case", "A 切法", "B 切法", "通信", "场景"),
    ("1", "I_X, J",     "J, K_Y",     "无",         "TP column→row 理想"),
    ("2", "I, J_X",     "J, K",       "all-gather A", "FSDP gather 参数"),
    ("3", "I, J_X",     "J_X, K",     "all-reduce C", "TP row parallel"),
    ("4", "I_X, J",     "J, K_X",     "无 (输出已切)", "TP column parallel 输出"),
]
widths = [6, 14, 14, 18, 30]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "Case":
        print("-" * len(line))

print()
print("记忆窍门：contracting 维度（J）的切分情况决定通信需求。")
print("  - A、B 都不切 J → 无通信")
print("  - 一方切 J → 需要先 all-gather 去掉这个切分")
print("  - 双方都切 J → 局部得到 partial sum，需要 all-reduce")


## 12. forward/backward 对偶：省一半记忆

理解了 partition notation 和 matmul 四种 case 后，有一个有用的观察：**集体通信在 forward 和 backward 之间是对偶的**。掌握对偶关系，记一遍等于记两遍。

**对偶规则一：all-gather ↔ reduce-scatter**

前向用 all-gather 把分片拼成完整张量，反向对应的操作是 reduce-scatter——把上游梯度按对应切片分发并求和。原因：all-gather 在前向把同一片复制到每个设备，每个设备参与不同的下游计算；反向时这些不同计算的梯度要汇聚回原始那一片，正好是 reduce-scatter。

数学上，如果 $y = a + b$，则 $\partial f / \partial x = \partial f / \partial a \cdot \partial a / \partial x + \partial f / \partial b \cdot \partial b / \partial x$。多个分支的梯度求和后传给上游，这正是 reduce-scatter。

**对偶规则二：fan-out ↔ sum reduction**

前向里「一个输入复制到多个分支」（fan-out），反向里对应「多个分支梯度求和」（sum reduction）。等价于规则一。

**对偶规则三：reduce-scatter ↔ all-gather**

前向用 reduce-scatter 把多个输入聚合成一个切片，反向对应 all-gather——把一个切片的梯度广播给所有贡献者。

**推论：all-reduce 的反向还是 all-reduce**

all-reduce = reduce-scatter + all-gather。反向 = all-gather 的反向 + reduce-scatter 的反向 = reduce-scatter + all-gather = all-reduce。所以 DDP 反向时 all-reduce 梯度，反向的反向（如果有的话）还是 all-reduce。

掌握这套对偶后，看任何并行方案的 forward，就能推出 backward 的通信模式，不需要单独记 backward。


In [ ]:
# === forward/backward 对偶示意 ===
print("=== 集体通信的 forward / backward 对偶表 ===\n")
pairs = [
    ("forward 通信", "backward 通信", "直觉"),
    ("all-gather",      "reduce-scatter",  "前向拼起来，反向切回去求和"),
    ("reduce-scatter",  "all-gather",      "前向切并求和，反向拼回去"),
    ("broadcast",       "reduce",          "前向 1→all，反向 all→1 求和"),
    ("all-reduce",      "all-reduce",      "自对偶（reduce-scatter + all-gather）"),
    ("fan-out (复制)",   "sum reduction",   "前向复制到 N 分支，反向 N 个梯度求和"),
]
widths = [22, 22, 40]
for row in pairs:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "forward 通信":
        print("-" * len(line))

print("\n记忆价值：看到论文里写 forward 的通信模式，反向不用再推一遍，直接查表。")
print("举例：FSDP 前向 all-gather 参数 → 反向 reduce-scatter 梯度。")
print("     TP column parallel 前向无通信 → 反向也无通信（输出已切分）。")


## 13. torch.distributed：真实的低级 API

前面用 numpy 在单进程里模拟了 collective 的语义。真实分布式训练里用的是 `torch.distributed` 模块（简称 `dist`），它通过 NCCL（GPU 间）或 gloo（CPU 间）backend 调用底层高效实现。

几个核心 API 的签名：

- `dist.init_process_group(backend, init_method, rank, world_size)`：初始化进程组，所有进程必须先调用这个才能用其他 collective。
- `dist.all_reduce(tensor, op=ReduceOp.SUM, group)`：原地 all-reduce，tensor 直接变成聚合结果。
- `dist.all_gather(tensor_list, tensor, group)`：把每张卡的 tensor 收集到 tensor_list（不是原地，输出长度 = world_size）。
- `dist.reduce_scatter(output, input_list, op, group)`：聚合 input_list 后切片到 output。
- `dist.all_to_all_single(output, input, group)`：all-to-all 的 single-tensor 版本，常用于 MoE 路由。

下面用 gloo backend 在单机启动 2 个进程，真实跑一次 all-reduce。需要 `torch.multiprocessing.spawn` 把同一个函数在两个子进程里跑起来。


In [ ]:
# === torch.distributed 真实 2-process demo ===
# 这段代码在单机上用 multiprocessing 启动 2 个进程，通过 gloo backend 做 all-reduce
import os
import torch
import torch.multiprocessing as mp
import torch.distributed as dist


def worker_fn(rank, world_size):
    """每个子进程跑这个函数。rank 是自己的编号，world_size 是总进程数。"""
    # 1. 初始化进程组
    # 用 gloo backend（CPU 间通信，不需要 GPU）
    # env:// 表示从环境变量读 init_method
    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = "29501"
    dist.init_process_group(
        backend="gloo",
        rank=rank,
        world_size=world_size,
    )

    # 2. 每个进程持有一个 tensor
    # rank 0 持有 [1.0, 2.0]，rank 1 持有 [2.0, 3.0]
    tensor = torch.tensor([float(rank + 1), float(rank + 2)])
    print(f"[before] rank {rank}: {tensor.tolist()}")

    # 3. all-reduce sum
    # 注意：是原地操作，tensor 直接被改写
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)

    # 两个 rank 对应位置相加：[1+2, 2+3] = [3, 5]
    print(f"[after]  rank {rank}: {tensor.tolist()}  (期望 [3.0, 5.0])")

    # 4. 清理
    dist.destroy_process_group()


# 启动 2 个进程
if __name__ == "__main__":
    world_size = 2
    mp.spawn(worker_fn, args=(world_size,), nprocs=world_size, join=True)
    print("\n所有进程结束。两个 rank 的 all-reduce 结果都是 [3.0, 5.0]。")


In [ ]:
# === torch.distributed 核心 API 签名（伪代码展示，不实际调用） ===
api_reference = '''
# 初始化（每个进程启动时调用一次）
dist.init_process_group(
    backend="nccl" | "gloo",      # nccl 用于 GPU，gloo 用于 CPU
    init_method="env://",          # 也可 file:// 或 tcp://
    rank=int,                      # 当前进程编号
    world_size=int,                # 总进程数
)

# all-reduce：原地修改 tensor
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)   # op 默认 SUM
# 支持的 op: SUM, PRODUCT, MIN, MAX, BAND, BOR, BXOR

# all-gather：输出是 list，长度 = world_size
output_list = [torch.empty_like(tensor) for _ in range(world_size)]
dist.all_gather(output_list, tensor)

# reduce-scatter：输入是 list，输出是单 tensor
input_list = [torch.rand(4) for _ in range(world_size)]
output = torch.empty(4)
dist.reduce_scatter(output, input_list, op=dist.ReduceOp.SUM)

# broadcast：从 src rank 广播到所有
dist.broadcast(tensor, src=0)

# all-to-all（single tensor 版本，用于 MoE）
output = torch.empty(input.shape)
dist.all_to_all_single(output, input)

# 同步屏障：所有进程到达后才能继续
dist.barrier()
'''
print(api_reference)
print("记忆要点：")
print("  - all_reduce / broadcast / all_to_all_single 是原地操作")
print("  - all_gather / reduce_scatter 用 list 中转")
print("  - 所有 op 默认 SUM，DDP 同步梯度用这个")


## 14. MoE 的 all-to-all：Expert Parallelism 的瓶颈

Mixture of Experts（MoE）模型里，每个 token 被路由到 top-k 个 expert 上计算。当 expert 数量很多（比如 DeepSeek-MoE 有 256 个 expert），单卡放不下所有 expert，就把 expert 分散到多张卡上——这就是 Expert Parallelism（EP）。

EP 下，每张卡持有 $E/N$ 个 expert（$E$ 是总 expert 数，$N$ 是 EP 并行度）。一个 batch 的 token 在每张卡上经过 router 后，需要被分发到持有对应 expert 的卡上计算，计算完再收回来。这两次分发和收回，都是 all-to-all。

**前向的通信模式**：

1. 每张卡持有本地 batch 的 token，router 决定每个 token 去哪些 expert。
2. **dispatch all-to-all**：把 token 按「目标 expert 所在的卡」分组，发给对应卡。每张卡收到来自所有其他卡的 token，拼成「将要被自己持有的 expert 处理的 token 集合」。
3. 本地 expert 计算（标准 FFN）。
4. **combine all-to-all**：把每个 expert 的输出按「token 来源」分组，发回原卡。每张卡收到自己原始 token 的对应输出，加回到 residue stream。

**通信量**：每次 all-to-all 的数据量 ≈ batch_size × seq_len × hidden_dim × top_k。在 Mixtral 8×7B 这种规模上，这个数能到几 GB，是 EP 训练的主要通信开销。

**为什么 all-to-all 是 EP 的瓶颈**：和 all-reduce 不同，all-to-all 无法用 ring 算法优化到「通信量与 N 无关」。每张卡发给每张卡的数据都不同，本质上是 $N^2$ 条点对点通信。这就是为什么 EP 的并行度通常比 DP/TP 小很多——Mixtral 训练时 EP=8 已经是较大规模，再大就会被 all-to-all 卡住。


In [ ]:
# === MoE EP 的 all-to-all 通信模拟 ===
import numpy as np

N = 4   # 4 张卡，每张卡持有 2 个 expert（共 8 个 expert）
E_per_card = 2
batch_per_card = 4   # 每张卡输入 4 个 token
hidden = 8

np.random.seed(0)
# 每张卡的输入 token（模拟），形状 (batch_per_card, hidden)
tokens = [np.random.randn(batch_per_card, hidden) for _ in range(N)]

# router 决定每个 token 去哪个 expert（这里简化为 1 个目标 expert）
# 真实 MoE 是 top-k，这里 top-1 看清楚 dispatch 过程
router_decisions = [np.random.randint(0, N * E_per_card, size=batch_per_card)
                    for _ in range(N)]

print("=== dispatch 阶段：每张卡按 router 把 token 发给对应 expert 所在的卡 ===\n")
for src in range(N):
    print(f"rank {src} 的 token 路由：{router_decisions[src]}")
    # expert_id // E_per_card = 目标卡号
    targets = router_decisions[src] // E_per_card
    print(f"  → 目标卡号: {targets.tolist()}")

# 模拟 dispatch all-to-all：构建每张卡的 send buffer
# send_buf[dst] 是要发给 dst 卡的 token 列表
print("\n=== dispatch 后：每张卡收到的 token ===")
recv_tokens = [[] for _ in range(N)]
for src in range(N):
    targets = router_decisions[src] // E_per_card
    for i, dst in enumerate(targets):
        recv_tokens[dst].append((src, tokens[src][i], router_decisions[src][i]))

for dst in range(N):
    print(f"rank {dst} 收到 {len(recv_tokens[dst])} 个 token：")
    for src, tok, eid in recv_tokens[dst]:
        print(f"  来自 rank {src}，目标 expert {eid}（本地 idx {eid - dst * E_per_card}）")

print("\n=== combine 阶段：expert 计算后，结果按原路 all-to-all 发回 ===")
print("combine 是 dispatch 的反向：每张卡把每个 expert 的输出按 token 来源发回原卡。")
print("两次 all-to-all 的通信量都 = N × batch × hidden × top_k。")
print("\n关键观察：EP 的通信开销集中在两次 all-to-all，无法用 ring 优化。")
print("DeepSeek-MoE 通过 shared expert + 细粒度 expert 缓解，但 all-to-all 仍是瓶颈。")


In [ ]:
# === MoE EP 的通信量手算 ===
# 以 Mixtral 8x7B 为例
batch_size = 1024        # 全局 batch
seq_len = 4096
hidden = 4096
top_k = 2
N_ep = 8                 # EP 并行度
dtype_bytes = 2          # BF16

# 每次 all-to-all 的通信量（每张卡发出/收到的数据）
# 每张卡发出的 token 数 ≈ batch × seq / N_ep × top_k（分散到 N_ep 张卡）
tokens_per_card = batch_size * seq_len / N_ep
# 每个 token 是 hidden 维向量
bytes_per_send = tokens_per_card * top_k * hidden * dtype_bytes
gb_per_send = bytes_per_send / 1e9

print(f"Mixtral 8x7B 配置（EP={N_ep}，top_k={top_k}）：")
print(f"  每张卡每次 all-to-all 发送：{gb_per_send:.2f} GB")
print(f"  每个 MoE 层有 2 次 all-to-all（dispatch + combine）：{2 * gb_per_send:.2f} GB")
print(f"  32 层 MoE 的总通信量：{32 * 2 * gb_per_send:.1f} GB")

print("\n对比：相同配置下的 all-reduce 通信量（DDP 反向）")
ddp_data = batch_size * seq_len * hidden * dtype_bytes / 1e9
print(f"  一次 all-reduce ≈ {ddp_data:.2f} GB × 2(N-1)/N ≈ {ddp_data * 2 * (N_ep-1)/N_ep:.2f} GB")
print(f"  ring 算法让 DDP 通信量与 N_ep 几乎无关，但 EP 的 all-to-all 无法受益。")

print("\n结论：EP 训练的通信瓶颈在 all-to-all，而非 all-reduce。")
print("这就是为什么 EP 并行度通常远小于 DP/TP——all-to-all 的代价随 N 增长。")


## 小结

确认你已经搞懂了这些：

- [ ] send/recv 是点对点通信，阻塞版本容易死锁；process group 是 collective 的作用域
- [ ] broadcast 是 1→all 复制，scatter 是 1→all 切片，gather 是 all→1 拼接
- [ ] reduce 是 all→1 聚合，all-reduce 是 all→all 聚合（每卡都有完整结果）
- [ ] all-gather 是 all→all 拼接，reduce-scatter 是 all→all 聚合切片
- [ ] all-to-all 是 all↔all 重排，无法用 ring 优化，通信量随 N 平方级
- [ ] ring all-reduce 把通信量降到 2(N-1)/N × D，与设备数几乎无关
- [ ] partition notation：下标出现 = 切分，不出现 = 复制；$\{U_X\}$ 表示未 reduce
- [ ] 分块 matmul 四种 case：contracting 维度切分情况决定是否需要通信
- [ ] forward/backward 对偶：all-gather ↔ reduce-scatter，all-reduce 自对偶
- [ ] torch.distributed 的 all_reduce/all_gather/reduce_scatter/all_to_all_single 签名
- [ ] MoE EP 的瓶颈是两次 all-to-all（dispatch + combine），无法用 ring 优化


## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：手算 ring all-reduce 在 N=8 时的通信量**

一个大小为 64 MB 的 tensor 在 8 张卡上做 ring all-reduce（sum）。每张卡发送和接收的总数据量是多少 MB？朴素 all-reduce（rank 0 收集再广播）又是多少？

小提示：ring all-reduce 的两个阶段各 N-1 轮，每轮每卡发 D/N。朴素实现每卡发送 D，rank 0 接收 (N-1)×D。


In [ ]:
# 作业 1：ring all-reduce vs 朴素 all-reduce 通信量
D_mb = 64    # tensor 大小 64 MB
N = 8        # 8 张卡

# TODO: 计算两种实现的每卡总通信量（MB）
ring_per_card = None     # ring all-reduce 每卡发送+接收总量
naive_per_card = None    # 朴素 all-reduce（rank 0 收集再广播）每卡最大通信量

assert ring_per_card is not None and naive_per_card is not None, "请先计算两个通信量"
# ring all-reduce = reduce-scatter + all-gather
# 每个阶段每卡发送 (N-1) × D/N，两个阶段共 2(N-1) × D/N
expected_ring = 2 * (N - 1) * D_mb / N
# 朴素实现：rank 0 接收 (N-1)×D 发送 D×(N-1)？不对
# 朴素：每张非-0 卡发送 D 给 rank 0；rank 0 接收 (N-1)×D，再广播发送 (N-1)×D
# 取最大值是 rank 0 的通信量
expected_naive = 2 * (N - 1) * D_mb   # rank 0 收 (N-1)*D + 发 (N-1)*D

assert abs(ring_per_card - expected_ring) < 0.1, f"ring 应为 {expected_ring:.1f} MB"
assert abs(naive_per_card - expected_naive) < 0.1, f"朴素最大值应为 {expected_naive:.1f} MB"

print(f"作业 1 通过：")
print(f"  ring all-reduce 每卡通信量：{ring_per_card:.1f} MB")
print(f"  朴素 all-reduce 最大通信量：{naive_per_card:.1f} MB（rank 0 是瓶颈）")
print(f"  ring 是朴素的 {ring_per_card / naive_per_card * 100:.1f}%，N 越大优势越明显。")


**作业 2：用 partition notation 描述 ZeRO Stage 3 的前向通信**

ZeRO Stage 3（FSDP）在前向传播时，每个 layer 的参数被分片到 N 张卡上，使用前先 all-gather 拼起来。假设参数矩阵 $W$ 沿 $X$ 轴分片（$W[I, J_X]$），写出前向 all-gather 的 partition notation 形式。

小提示：参考第 10.1 节的 all-gather 公式 $\text{AllGather}_Y: A[I, J_Y] \to A[I, J]$，把 $Y$ 换成 $X$。


In [ ]:
# 作业 2：ZeRO Stage 3 前向的 partition notation
# 填空：写出 FSDP 前向 all-gather 的输入和输出 partition 形式

# 输入参数 W 的 partition 形式（沿 X 切分）：
fsdp_input_notation = None    # 填字符串，形如 "W[I, JX]"

# all-gather 后的 partition 形式（不再切分）：
fsdp_output_notation = None   # 填字符串，形如 "W[I, J]"

assert fsdp_input_notation == "W[I, J_X]", \
    "FSDP 前向前参数沿 X 切片，应为 W[I, J_X]"
assert fsdp_output_notation == "W[I, J]", \
    "all-gather 去掉 X 切分，应为 W[I, J]"

print("作业 2 通过：")
print(f"  前向前：{fsdp_input_notation}（每卡持有 1/N 的参数）")
print(f"  all-gather 后：{fsdp_output_notation}（每卡临时持有完整参数）")
print(f"  反向对应 reduce-scatter：{fsdp_output_notation} → {fsdp_input_notation}（梯度切片后聚合）")
print("\n关键观察：FSDP 的通信模式就是 all-gather（前向）+ reduce-scatter（反向），互为对偶。")


**作业 3：计算 MoE EP 在不同并行度下的 all-to-all 通信量**

Mixtral 8×7B，batch_size=512，seq_len=2048，hidden=4096，top_k=2，BF16。分别计算 EP 并行度为 4 和 8 时，每个 MoE 层单卡的 all-to-all 通信量（GB）。结果说明 EP 并行度增加时通信量如何变化？

小提示：每卡每次 all-to-all 发送 = (batch × seq / N_ep) × top_k × hidden × 2 bytes。一层有 2 次 all-to-all。


In [ ]:
# 作业 3：MoE EP 的 all-to-all 通信量
batch = 512
seq = 2048
hidden = 4096
top_k = 2
bytes_per_elem = 2  # BF16

def ep_all_to_all_gb(N_ep):
    """计算一层 MoE 单卡 all-to-all 总通信量（GB）。"""
    # TODO: 在这里实现
    # 每卡每次 all-to-all 发送的数据量
    # 一层有 2 次 all-to-all（dispatch + combine）
    return None

# 测试两种并行度
comm_4 = ep_all_to_all_gb(4)
comm_8 = ep_all_to_all_gb(8)

assert comm_4 is not None and comm_8 is not None, "请先实现 ep_all_to_all_gb"

# 验证：每卡每次 all-to-all = (batch*seq/N_ep) * top_k * hidden * 2 bytes
# 一层 2 次
def expected(N_ep):
    tokens_per_card = batch * seq / N_ep
    bytes_per_send = tokens_per_card * top_k * hidden * bytes_per_elem
    return 2 * bytes_per_send / 1e9   # 一层 2 次

assert abs(comm_4 - expected(4)) < 0.01, f"N_ep=4 时应为 {expected(4):.2f} GB"
assert abs(comm_8 - expected(8)) < 0.01, f"N_ep=8 时应为 {expected(8):.2f} GB"

print(f"作业 3 通过：")
print(f"  EP=4 时一层 MoE 单卡 all-to-all：{comm_4:.2f} GB")
print(f"  EP=8 时一层 MoE 单卡 all-to-all：{comm_8:.2f} GB")
print(f"  并行度翻倍后，通信量{'减少' if comm_8 < comm_4 else '增加'}到原来的 {comm_8/comm_4*100:.1f}%")
print()
print("关键观察：EP 并行度增加，每卡 all-to-all 通信量按 1/N 递减。")
print("但 all-to-all 的延迟随 N 增长（更多跳点对点），所以 EP 并行度不是越大越好。")


## 参考资料

- [NCCL Documentation](https://docs.nvidia.com/deeplearning/nccl/) — NVIDIA 的 GPU 集合通信库，all-reduce/all-gather 的工业实现
- [PyTorch torch.distributed](https://pytorch.org/docs/stable/distributed.html) — 分布式训练的官方 API
- Sergeev & Del Balso, [Horovod: fast and easy distributed deep learning in TensorFlow](https://arxiv.org/abs/1802.05799), 2018 — ring all-reduce 的工程实现参考
- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020 — ZeRO Stage 1/2/3，用到 reduce-scatter 和 all-gather
- Shoeybi et al., [Megatron-LM](https://arxiv.org/abs/1909.08053), 2019 — tensor parallelism，用到 partition notation 和 all-reduce
- Fedus et al., [Switch Transformers](https://arxiv.org/abs/2101.03961), 2021 — MoE expert parallelism 的 all-to-all 通信
- Jiang et al., [DeepSeek-MoE](https://arxiv.org/abs/2401.06066), 2024 — 细粒度 expert + shared expert 缓解 all-to-all 瓶颈
